In [13]:
# AI Investment Opportunity Analyzer
# Notebook 03: Preprocessing and Feature Engineering

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import joblib

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
MODELS_PATH = PROJECT_ROOT / "models"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

# Load dataset
data_file = RAW_DATA_PATH / "synthetic_investment_opportunities.csv"
df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5000, 42)


,opportunity_id,opportunity_name,sector,region,competition_level,investment_size_million,capex_million,opex_million,expected_roi_percent,irr_percent,payback_period_years,revenue_growth_percent,profit_margin_percent,npv_million,market_size_billion,market_growth_percent,demand_score,customer_adoption_score,scalability_score,financial_risk_score,regulatory_risk_score,execution_risk_score,market_risk_score,dependency_risk_score,overall_risk_score,vision_2030_alignment_score,economic_diversification_score,job_creation_score,localization_score,quality_of_life_score,strategic_impact_score,environmental_impact_score,social_impact_score,governance_score,sustainability_score,risk_adjusted_roi,profitability_index,payback_efficiency,market_attractiveness_score,financial_strength_score,investment_score,recommendation
0,INV-00001,Tabuk Technology Opportunity,Technology,Tabuk,Low,163.15,91.34,22.63,14.29,17.19,11.70,6.75,8.15,37.19,9.97,3.06,81.65,69.31,68.94,85.81,32.65,53.71,15.70,56.95,48.96,76.84,68.45,100.00,68.27,92.31,81.17,95.59,53.23,83.81,77.54,0.2860,0.2266,1.1252,58.80,40.56,80.10,Invest
1,INV-00002,Asir Healthcare Opportunity,Healthcare,Asir,Medium,67.46,47.73,11.44,19.08,20.73,5.18,7.90,40.00,49.05,15.38,1.28,23.04,79.19,91.22,29.95,17.16,20.65,49.88,42.50,32.03,82.05,100.00,84.51,96.38,91.63,90.91,91.58,65.18,64.37,73.71,0.5777,0.7165,3.0874,49.96,62.20,90.23,Invest
2,INV-00003,Eastern Province Education Opportunity,Education,Eastern Province,High,213.29,165.94,12.03,8.38,7.92,6.69,8.79,19.71,20.20,11.90,9.34,86.52,60.60,86.64,26.35,23.35,40.05,61.84,67.30,43.78,58.19,96.47,54.80,71.68,22.05,60.64,84.57,18.88,54.31,52.59,0.1871,0.0943,1.0897,70.11,36.20,22.60,Reject
3,INV-00004,Madinah Logistics Opportunity,Logistics,Madinah,Medium,23.72,19.82,3.41,16.41,18.17,3.30,11.64,28.78,26.97,9.28,14.04,65.22,22.05,65.52,52.23,39.64,57.27,58.35,37.99,49.10,85.49,80.36,68.48,56.42,54.87,69.12,19.74,54.31,70.71,48.25,0.3275,1.0910,3.8163,55.75,52.93,76.81,Invest
4,INV-00005,Qassim Technology Opportunity,Technology,Qassim,High,131.04,84.34,21.33,9.21,9.21,5.91,12.61,21.53,68.98,31.57,8.13,58.12,70.88,42.25,47.58,47.07,82.98,53.51,35.38,53.30,70.13,57.04,100.00,94.91,50.99,74.61,100.00,54.64,63.07,72.57,0.1696,0.5224,1.3329,52.97,38.86,32.86,Reject


In [14]:
# Columns that should not be used as model input

id_columns = [
    "opportunity_id",
    "opportunity_name"
]

target_columns = [
    "investment_score",
    "recommendation"
]

drop_columns = id_columns + target_columns

print("Columns excluded from model input:")
drop_columns

Columns excluded from model input:


['opportunity_id', 'opportunity_name', 'investment_score', 'recommendation']

In [15]:
# Define features and target

X = df.drop(columns=drop_columns)
y = df["investment_score"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature matrix shape: (5000, 38)
Target shape: (5000,)


,sector,region,competition_level,investment_size_million,capex_million,opex_million,expected_roi_percent,irr_percent,payback_period_years,revenue_growth_percent,profit_margin_percent,npv_million,market_size_billion,market_growth_percent,demand_score,customer_adoption_score,scalability_score,financial_risk_score,regulatory_risk_score,execution_risk_score,market_risk_score,dependency_risk_score,overall_risk_score,vision_2030_alignment_score,economic_diversification_score,job_creation_score,localization_score,quality_of_life_score,strategic_impact_score,environmental_impact_score,social_impact_score,governance_score,sustainability_score,risk_adjusted_roi,profitability_index,payback_efficiency,market_attractiveness_score,financial_strength_score
0,Technology,Tabuk,Low,163.15,91.34,22.63,14.29,17.19,11.70,6.75,8.15,37.19,9.97,3.06,81.65,69.31,68.94,85.81,32.65,53.71,15.70,56.95,48.96,76.84,68.45,100.00,68.27,92.31,81.17,95.59,53.23,83.81,77.54,0.2860,0.2266,1.1252,58.80,40.56
1,Healthcare,Asir,Medium,67.46,47.73,11.44,19.08,20.73,5.18,7.90,40.00,49.05,15.38,1.28,23.04,79.19,91.22,29.95,17.16,20.65,49.88,42.50,32.03,82.05,100.00,84.51,96.38,91.63,90.91,91.58,65.18,64.37,73.71,0.5777,0.7165,3.0874,49.96,62.20
2,Education,Eastern Province,High,213.29,165.94,12.03,8.38,7.92,6.69,8.79,19.71,20.20,11.90,9.34,86.52,60.60,86.64,26.35,23.35,40.05,61.84,67.30,43.78,58.19,96.47,54.80,71.68,22.05,60.64,84.57,18.88,54.31,52.59,0.1871,0.0943,1.0897,70.11,36.20
3,Logistics,Madinah,Medium,23.72,19.82,3.41,16.41,18.17,3.30,11.64,28.78,26.97,9.28,14.04,65.22,22.05,65.52,52.23,39.64,57.27,58.35,37.99,49.10,85.49,80.36,68.48,56.42,54.87,69.12,19.74,54.31,70.71,48.25,0.3275,1.0910,3.8163,55.75,52.93
4,Technology,Qassim,High,131.04,84.34,21.33,9.21,9.21,5.91,12.61,21.53,68.98,31.57,8.13,58.12,70.88,42.25,47.58,47.07,82.98,53.51,35.38,53.30,70.13,57.04,100.00,94.91,50.99,74.61,100.00,54.64,63.07,72.57,0.1696,0.5224,1.3329,52.97,38.86


In [16]:
# Identify categorical and numerical features

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))

print("\nNumber of numerical features:", len(numerical_features))
print("\nNumerical features:")
print(numerical_features)

Categorical features:
['sector', 'region', 'competition_level']

Number of categorical features: 3

Number of numerical features: 35

Numerical features:
['investment_size_million', 'capex_million', 'opex_million', 'expected_roi_percent', 'irr_percent', 'payback_period_years', 'revenue_growth_percent', 'profit_margin_percent', 'npv_million', 'market_size_billion', 'market_growth_percent', 'demand_score', 'customer_adoption_score', 'scalability_score', 'financial_risk_score', 'regulatory_risk_score', 'execution_risk_score', 'market_risk_score', 'dependency_risk_score', 'overall_risk_score', 'vision_2030_alignment_score', 'economic_diversification_score', 'job_creation_score', 'localization_score', 'quality_of_life_score', 'strategic_impact_score', 'environmental_impact_score', 'social_impact_score', 'governance_score', 'sustainability_score', 'risk_adjusted_roi', 'profitability_index', 'payback_efficiency', 'market_attractiveness_score', 'financial_strength_score']


C:\Users\iazoq\AppData\Local\Temp\ipykernel_31292\2802600803.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [17]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4000, 38)
X_test: (1000, 38)
y_train: (4000,)
y_test: (1000,)


In [18]:
# Build preprocessing pipeline

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [19]:
# Fit preprocessor on training data and transform train/test sets

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

X_train_processed shape: (4000, 56)
X_test_processed shape: (1000, 56)


In [20]:
# Get feature names after preprocessing

encoded_categorical_features = (
    preprocessor
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(categorical_features)
    .tolist()
)

processed_feature_names = numerical_features + encoded_categorical_features

print("Number of processed features:", len(processed_feature_names))
processed_feature_names[:10]

Number of processed features: 56


['investment_size_million',
 'capex_million',
 'opex_million',
 'expected_roi_percent',
 'irr_percent',
 'payback_period_years',
 'revenue_growth_percent',
 'profit_margin_percent',
 'npv_million',
 'market_size_billion']

In [21]:
# Convert processed arrays to DataFrames

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

print("Processed train shape:", X_train_processed_df.shape)
print("Processed test shape:", X_test_processed_df.shape)

X_train_processed_df.head()

Processed train shape: (4000, 56)
Processed test shape: (1000, 56)


,investment_size_million,capex_million,opex_million,expected_roi_percent,irr_percent,payback_period_years,revenue_growth_percent,profit_margin_percent,npv_million,market_size_billion,market_growth_percent,demand_score,customer_adoption_score,scalability_score,financial_risk_score,regulatory_risk_score,execution_risk_score,market_risk_score,dependency_risk_score,overall_risk_score,vision_2030_alignment_score,economic_diversification_score,job_creation_score,localization_score,quality_of_life_score,strategic_impact_score,environmental_impact_score,social_impact_score,governance_score,sustainability_score,risk_adjusted_roi,profitability_index,payback_efficiency,market_attractiveness_score,financial_strength_score,sector_Education,sector_Entertainment,sector_Healthcare,sector_Infrastructure,sector_Logistics,sector_Real Estate,sector_Renewable Energy,sector_Retail,sector_Technology,sector_Tourism,region_Asir,region_Eastern Province,region_Jazan,region_Madinah,region_Makkah,region_Qassim,region_Riyadh,region_Tabuk,competition_level_High,competition_level_Low,competition_level_Medium
4227,0.810231,1.180062,1.031915,-1.200437,-0.956338,-0.266855,0.183919,-1.600223,-0.911095,1.096239,-0.730957,0.547911,-0.880988,-0.303706,-0.688338,-0.276754,0.614126,-1.696383,-1.237674,-1.493458,0.566303,1.335540,1.015522,0.432644,0.537474,1.686944,-1.468850,-1.534814,-0.613080,-2.098692,-0.858262,-1.058890,-0.890424,-0.807626,-1.760426,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4676,-0.368924,-0.332767,-0.245274,-0.626896,-0.941014,-1.856467,2.969838,0.585105,-0.340575,0.252151,-1.684595,0.940981,0.327634,-0.699995,0.059868,1.471862,-1.379761,-0.827944,0.038216,-0.250264,-0.574938,-0.773068,-0.797860,1.670573,-0.863958,-0.587497,-0.447906,-0.324479,0.171644,-0.374332,-0.557349,-0.480706,0.469577,-0.638288,-0.470387,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
800,-0.514388,-0.478720,-0.519780,0.210209,1.570605,-2.237650,-0.376035,-1.435651,-0.022529,0.773616,2.097213,-0.946095,-0.134575,2.038934,0.371873,0.609596,-0.352123,0.086612,0.738303,0.671264,-1.731731,-0.986763,-1.016159,-2.149475,0.548640,-2.249321,-0.021962,1.456679,-0.177313,0.718174,-0.115344,1.047115,2.857600,1.740511,0.148950,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3671,-0.661229,-0.672095,-0.643947,0.377630,0.256985,1.172717,-0.668873,-0.237751,-0.271341,-0.667656,-0.976529,-0.049691,0.098046,1.362435,-0.688338,-0.951078,-0.781844,0.007001,-0.032537,-1.087029,-0.909894,-1.527519,-0.281265,-0.924828,-0.094972,-1.557956,0.042732,-1.631078,-0.351506,-1.076784,0.791141,0.584262,-0.289782,0.043097,0.134133,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4193,0.140625,0.126265,-0.265215,1.121907,1.962044,1.586341,-0.777698,0.539204,0.564209,-0.962297,-2.030442,1.167224,-0.943695,-0.350138,2.084871,-0.730081,-1.359797,-0.538626,-0.698644,-0.598011,0.305516,0.417842,0.029077,0.646758,-0.927406,0.161772,-0.393129,0.664706,-0.134479,0.061611,1.231273,0.634926,-0.057567,-1.009219,1.659262,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [22]:
# Save processed train/test datasets

X_train_processed_df.to_csv(PROCESSED_DATA_PATH / "X_train_processed.csv", index=False)
X_test_processed_df.to_csv(PROCESSED_DATA_PATH / "X_test_processed.csv", index=False)

y_train.to_csv(PROCESSED_DATA_PATH / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DATA_PATH / "y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [23]:
# Save fitted preprocessor

preprocessor_file = MODELS_PATH / "preprocessor.pkl"

joblib.dump(preprocessor, preprocessor_file)

print(f"Preprocessor saved to: {preprocessor_file}")

Preprocessor saved to: d:\Senior last\ML\ai-investment-opportunity-analyzer\models\preprocessor.pkl
